##  Import Libraries

In [1]:
# sklearn is a machine learning library
# we use it to build our churn prediction model
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries ready!")

Libraries ready!


## Load & Prepare Data

In [2]:
df = pd.read_csv('../data/churn_cleaned.csv')

# Model only understands numbers
# So convert Churn Yes/No to 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Select only number columns
# because model cant understand text columns
df_model = df.select_dtypes(include=['number']).copy()

print("Data ready!")
print("Columns used:", df_model.columns.tolist())

Data ready!
Columns used: ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Revenue at Risk']


## Split Data into Train & Test

In [3]:
# X = all columns except Churn (these are inputs)
# y = Churn column (this is what we want to predict)
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Split data into:
# 80% training — model learns from this
# 20% testing  — we test how good model is on this
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    # 20% for testing
    random_state=42)  # 42 means same split every time

print("Training rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])

Training rows: 5634
Testing rows : 1409


## Train & Test Model

In [4]:
# Create Logistic Regression model
# This model predicts Yes or No (Churn or Not Churn)
model = LogisticRegression(max_iter=1000)

# Train model on training data
model.fit(X_train, y_train)

# Test model on testing data
y_pred = model.predict(X_test)

# Check how accurate the model is
accuracy = round(accuracy_score(y_test, y_pred) * 100, 2)

# ROC-AUC score — higher is better
# 0.5 = random guessing, 1.0 = perfect
roc_auc = round(roc_auc_score(y_test, y_pred), 2)

print("=" * 40)
print("MODEL PERFORMANCE")
print("=" * 40)
print(f"Accuracy : {accuracy}%")
print(f"ROC-AUC  : {roc_auc}")

MODEL PERFORMANCE
Accuracy : 100.0%
ROC-AUC  : 1.0


## Give Each Customer a Risk Score

In [6]:
# predict_proba gives probability of churn
# between 0 and 1 for each customer
churn_prob = model.predict_proba(X_test)[:, 1]

# Categorize each customer as High/Medium/Low risk
def risk_category(prob):
    if prob >= 0.7:
        return 'High Risk'
    elif prob >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

# Convert numpy array to pandas Series first
# then we can use .apply() on it
churn_prob_series = pd.Series(churn_prob)

# Create final results table
results = X_test.copy().reset_index(drop=True)
results['Churn Probability'] = churn_prob_series.round(2)
results['Risk Category']     = churn_prob_series.apply(risk_category)
results['Actual Churn']      = y_test.values

# Show risk summary
print("=" * 40)
print("CHURN RISK SUMMARY")
print("=" * 40)
print(results['Risk Category'].value_counts())

CHURN RISK SUMMARY
Risk Category
Low Risk     1036
High Risk     373
Name: count, dtype: int64


##  Save & Final Summary

In [8]:
# Save risk scores to CSV
# This CSV can be used in Power BI dashboard
results.to_csv('../outputs/churn_risk_scores.csv', index=False)

# Final business numbers
high_risk = (results['Risk Category'] == 'High Risk').sum()
total     = len(results)

print("=" * 40)
print("FINAL BUSINESS SUMMARY")
print("=" * 40)
print(f"Total Customers Scored : {total}")
print(f"High Risk Customers    : {high_risk}")
print(f"Model Accuracy         : {accuracy}%")
print(f"ROC-AUC Score          : {roc_auc}")
print("\nRisk scores saved to outputs folder!")

FINAL BUSINESS SUMMARY
Total Customers Scored : 1409
High Risk Customers    : 373
Model Accuracy         : 100.0%
ROC-AUC Score          : 1.0

Risk scores saved to outputs folder!
